# Beta-Neutral Min-EDaR → Black-Litterman → Volatility Targeting

Research-grade long/short equity research notebook.

Implements:
- realistic 500-name synthetic US equity universe
- point-in-time-data hook for live research
- factor-model covariance `B F B' + D`
- Ledoit-Wolf residual shrinkage
- Bayesian/shrunk market beta
- market, style and sector neutrality bands
- block-bootstrap path scenarios
- exponential-cone EDaR optimization
- KKT-aware EDaR implied prior for Black-Litterman
- independent momentum/value/quality views
- posterior EDaR optimization
- EWMA volatility targeting with asymmetric deleveraging
- turnover, market-impact and borrow costs
- walk-forward backtesting
- ablations, parameter sensitivity and stress tests

This is a research template, not a claim of profitability or live-trading readiness.


In [ ]:
# Optional installs for a fresh environment:
# %pip install numpy pandas scipy matplotlib scikit-learn cvxpy yfinance statsmodels joblib
# Optional production exponential-cone solver:
# %pip install Mosek

import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from sklearn.covariance import LedoitWolf
from sklearn.linear_model import Ridge
from numpy.random import default_rng

RNG = default_rng(42)
TRADING_DAYS = 252


## 1. Configuration

The default research configuration is intentionally conservative:
- 500 names
- 3-year estimation window
- 2x gross exposure
- 2% single-name cap
- ±2% portfolio beta band
- ±1% style/sector exposure bands
- 10% annualized volatility target
- explicit turnover, impact and borrow costs


In [ ]:
@dataclass
class Config:
    lookback: int = 756
    cov_window: int = 504
    beta_window: int = 504
    alpha: float = 0.05
    gross_limit: float = 2.0
    position_limit: float = 0.02
    beta_band: float = 0.02
    factor_band: float = 0.01
    sector_band: float = 0.01
    tau: float = 0.05
    risk_aversion: float = 3.0
    target_vol: float = 0.10
    min_leverage: float = 0.25
    max_leverage: float = 2.0
    ewma_lambda: float = 0.94
    rebalance_days: int = 21
    turnover_bps: float = 5.0
    impact_bps: float = 10.0
    borrow_bps: float = 50.0
    n_z: int = 10
    solver: str = "CLARABEL"

CFG = Config()
N_ASSETS = 500
SECTORS = [
    "Technology","Financials","Healthcare","Industrials","Consumer",
    "Energy","Utilities","Materials","RealEstate","Communication"
]
FACTOR_NAMES = ["Market","Size","Value","Momentum","Quality","LowVol"]


## 2. Realistic synthetic universe

The synthetic universe has 500 liquid US-equity-like names with sector classifications, market caps, ADV, prices and factor exposures. This avoids pretending that a backtest based on today's constituents is point-in-time correct.

For real research, replace it with a survivorship-bias-free point-in-time universe.


In [ ]:
def make_universe(n=500, seed=42):
    rng = default_rng(seed)
    tickers = [f"STK{i:04d}" for i in range(n)]
    meta = pd.DataFrame(index=tickers)
    meta["Sector"] = rng.choice(SECTORS, n)
    meta["Beta"] = np.clip(rng.normal(1.0, 0.22, n), 0.45, 1.65)
    for f in ["Size","Value","Momentum","Quality","LowVol"]:
        meta[f] = rng.normal(0, 1, n)
    meta["ADV"] = np.exp(rng.normal(np.log(25e6), 0.9, n))
    meta["Price"] = np.exp(rng.normal(np.log(100), 0.55, n))
    meta["MarketCap"] = np.exp(rng.normal(np.log(15e9), 1.25, n))
    return meta

universe = make_universe(N_ASSETS)
universe.head()


## 3. Factor/stochastic-volatility simulator

The simulator contains persistent style and sector factors, stochastic volatility and clustered crisis periods. It is for unit/stress testing; live research should use point-in-time market data.


In [ ]:
def simulate_market(n_days=3200, meta=universe, seed=42):
    rng = default_rng(seed)
    dates = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=n_days)

    factors = pd.DataFrame(
        index=dates,
        columns=FACTOR_NAMES + [f"S_{s}" for s in SECTORS],
        dtype=float
    )
    factor_scales = {
        "Market": (0.00025,0.010),
        "Size": (0.00002,0.0045),
        "Value": (0.00005,0.0045),
        "Momentum": (0.00007,0.0055),
        "Quality": (0.00005,0.0035),
        "LowVol": (0.00004,0.0030),
    }
    for f,(m,s) in factor_scales.items():
        factors[f] = rng.normal(m,s,n_days)
    for s in SECTORS:
        factors[f"S_{s}"] = rng.normal(0,0.005,n_days)

    # Small factor persistence.
    for c in factors.columns:
        x = factors[c].values
        for t in range(1,n_days):
            x[t] += 0.05*x[t-1]

    vol_state = np.ones(n_days)
    for t in range(1,n_days):
        vol_state[t] = 0.97*vol_state[t-1] + 0.03*np.exp(rng.normal(0,0.35))

    # Clustered crisis regimes.
    starts = rng.choice(np.arange(200,n_days-150), 10, replace=False)
    for st in starts:
        L = int(rng.integers(10,35))
        end = min(st+L,n_days)
        factors.iloc[st:end, factors.columns.get_loc("Market")] += rng.normal(-0.010,0.020,end-st)
        vol_state[st:end] *= 2.5

    X = np.zeros((n_days,len(meta)))
    for j,a in enumerate(meta.index):
        row = meta.loc[a]
        X[:,j] = (
            row.Beta*factors["Market"].values
            + 0.003*row.Size*factors["Size"].values
            + 0.004*row.Value*factors["Value"].values
            + 0.004*row.Momentum*factors["Momentum"].values
            + 0.003*row.Quality*factors["Quality"].values
            + 0.002*row.LowVol*factors["LowVol"].values
            + factors[f"S_{row.Sector}"].values
        )

    idio = rng.uniform(0.010,0.022,len(meta))
    X += rng.normal(0,1,(n_days,len(meta))) * idio[None,:] * vol_state[:,None]

    return (
        pd.DataFrame(X,index=dates,columns=meta.index),
        factors["Market"].rename("Benchmark"),
        factors
    )

returns, benchmark, factor_returns = simulate_market()
print(returns.shape)


## 4. Exposure matrix and Bayesian/shrunk beta


In [ ]:
def exposure_matrix(meta):
    cols = FACTOR_NAMES + [f"S_{s}" for s in SECTORS]
    B = pd.DataFrame(index=meta.index, columns=cols, dtype=float)
    B["Market"] = meta["Beta"]
    for f in ["Size","Value","Momentum","Quality","LowVol"]:
        B[f] = meta[f]
    for s in SECTORS:
        B[f"S_{s}"] = (meta["Sector"] == s).astype(float)
    return B

def shrunk_betas(R, benchmark, window=504, shrink=0.25):
    R = R.iloc[-window:]
    b = benchmark.iloc[-window:]
    beta_hat = R.apply(lambda x: np.cov(x,b)[0,1] / np.var(b))
    return (1-shrink)*beta_hat + shrink*1.0

B = exposure_matrix(universe)


## 5. Factor-model covariance

Estimate factor returns with ridge regression and model residual covariance with Ledoit-Wolf shrinkage. The final covariance is:

`Sigma = B F B' + D`


In [ ]:
def factor_covariance(R, meta, window=504):
    R = R.iloc[-window:]
    B = exposure_matrix(meta).loc[R.columns]

    ridge = Ridge(alpha=10.0, fit_intercept=False)
    ridge.fit(B.values, R.values)

    fitted = ridge.predict(B.values)
    F = np.cov(fitted.T)

    residual = R.values - fitted
    lw = LedoitWolf().fit(residual)
    D = np.diag(np.maximum(np.diag(lw.covariance_),1e-8))

    S = B.values @ F @ B.values.T + D
    return pd.DataFrame(S,index=R.columns,columns=R.columns)

Sigma = factor_covariance(returns,universe,CFG.cov_window)
print(Sigma.shape)


## 6. EDaR utilities

EDaR is path dependent, so block bootstrap rather than iid resampling is used for scenario generation.


In [ ]:
def drawdowns(x):
    c = np.cumsum(np.asarray(x))
    return np.maximum.accumulate(c) - c

def edar_value(x, alpha=0.05):
    D = drawdowns(x)
    def objective(log_z):
        z = np.exp(log_z)
        a = D/z
        m = np.max(a)
        log_mean_exp = m + np.log(np.mean(np.exp(a-m)))
        return z*(log_mean_exp-np.log(alpha))
    res = minimize_scalar(
        objective,
        bounds=(np.log(1e-5),np.log(2.0)),
        method="bounded"
    )
    return float(res.fun),float(np.exp(res.x))

def block_bootstrap(R,n_paths=100,block=20,seed=42):
    rng = default_rng(seed)
    X = R.values
    T = len(X)
    out=[]
    for _ in range(n_paths):
        chunks=[]
        total=0
        while total<T:
            s=rng.integers(0,max(1,T-block))
            c=X[s:s+block]
            chunks.append(c)
            total += len(c)
        out.append(np.vstack(chunks)[:T])
    return np.asarray(out)


## 7. Convex fixed-z EDaR optimizer

For fixed `z`, the objective

`z * (log_sum_exp(D/z) - log(T) - log(alpha))`

is convex and exponential-cone representable. We grid over z and retain the best feasible solution.


In [ ]:
def solve_edar_fixed_z(
    R,z,beta,B_constraints,cfg=CFG,
    mu=None,delta=0.0,prev=None
):
    T,N=R.shape
    w=cp.Variable(N)
    cum=cp.cumsum(R@w)
    h=cp.Variable(T)

    cons=[h[0]>=cum[0]]
    for t in range(1,T):
        cons += [h[t]>=h[t-1], h[t]>=cum[t]]
    D=h-cum

    cons += [
        cp.sum(w)==0,
        cp.norm1(w)<=cfg.gross_limit,
        w<=cfg.position_limit,
        w>=-cfg.position_limit,
        beta@w<=cfg.beta_band,
        beta@w>=-cfg.beta_band
    ]

    for j in range(B_constraints.shape[1]):
        band = cfg.sector_band if B_constraints.columns[j].startswith("S_") else cfg.factor_band
        cons += [
            B_constraints.iloc[:,j].values@w<=band,
            B_constraints.iloc[:,j].values@w>=-band
        ]

    risk=z*(cp.log_sum_exp(D/z)-np.log(T)-np.log(cfg.alpha))

    if mu is None:
        objective = risk
    else:
        objective = risk-delta*(mu@w)

    if prev is not None:
        objective += 0.002*cp.norm1(w-prev)

    problem=cp.Problem(cp.Minimize(objective),cons)
    problem.solve(solver=cfg.solver,verbose=False)

    if w.value is None:
        return None

    ww=np.asarray(w.value).ravel()
    ev,_=edar_value(R@ww,cfg.alpha)
    return pd.Series(ww),ev,float(problem.value)

def min_edar(R,beta,B_constraints,cfg=CFG,prev=None):
    zs=np.geomspace(0.002,0.20,cfg.n_z)
    sols=[]
    for z in zs:
        s=solve_edar_fixed_z(
            R.values,z,beta.values,B_constraints,
            cfg=cfg,prev=prev
        )
        if s is not None:
            sols.append(s)
    if not sols:
        raise RuntimeError("No feasible EDaR solution.")
    best=min(sols,key=lambda x:x[1])
    w=best[0]
    w.index=R.columns
    return w,best


## 8. KKT-aware EDaR implied prior

For variance optimization the classical BL relation is `pi = delta Sigma w`. For EDaR it is not.

We therefore estimate `grad EDaR(w)` and construct a prior in the normal space of the active neutrality constraints. This is a practical numerical KKT approximation.


In [ ]:
def edar_gradient(w,R,eps=1e-5,alpha=0.05):
    base=edar_value(R.values@w.values,alpha)[0]
    g=np.zeros(len(w))
    for i in range(len(w)):
        wp=w.copy()
        wp.iloc[i]+=eps
        g[i]=(edar_value(R.values@wp.values,alpha)[0]-base)/eps
    return pd.Series(g,index=w.index)

def kkt_prior(w,R,beta,B_constraints,delta=3.0):
    g=edar_gradient(w,R)
    raw=delta*g
    A=np.column_stack([
        np.ones(len(w)),
        beta.values,
        B_constraints.values
    ])
    coef=np.linalg.lstsq(A,raw.values,rcond=None)[0]
    return pd.Series(A@coef,index=w.index),g


## 9. Independent Black-Litterman views

Views use a separate signal layer: medium-term momentum, value and quality. In production, this must be point-in-time and independently validated.


In [ ]:
def independent_views(R,meta):
    mom=R.iloc[-126:].mean()-R.iloc[-21:].mean()
    mom=(mom-mom.mean())/(mom.std()+1e-9)
    val=(meta["Value"]-meta["Value"].mean())/(meta["Value"].std()+1e-9)
    qual=(meta["Quality"]-meta["Quality"].mean())/(meta["Quality"].std()+1e-9)

    signal=0.015*mom+0.008*val+0.008*qual

    P=[]
    q=[]
    for a in signal.abs().nlargest(12).index:
        row=np.zeros(len(R.columns))
        row[list(R.columns).index(a)]=1
        P.append(row)
        q.append(signal[a])

    return pd.DataFrame(P,columns=R.columns),np.asarray(q)

def black_litterman(Sigma,pi,P,q,tau=0.05):
    S=Sigma.values
    Pm=P.values
    TS=tau*S
    Omega=np.diag(np.diag(Pm@TS@Pm.T))

    invTS=np.linalg.pinv(TS)
    invO=np.linalg.pinv(Omega)

    post_cov=np.linalg.pinv(invTS+Pm.T@invO@Pm)
    mu=post_cov@(invTS@pi.values+Pm.T@invO@q)

    return (
        pd.Series(mu,index=Sigma.index),
        pd.DataFrame(post_cov,index=Sigma.index,columns=Sigma.columns),
        Omega
    )


## 10. Posterior EDaR optimization


In [ ]:
def posterior_edar(R,mu,beta,B_constraints,cfg=CFG,prev=None):
    zs=np.geomspace(0.002,0.20,cfg.n_z)
    sols=[]
    for z in zs:
        s=solve_edar_fixed_z(
            R.values,z,beta.values,B_constraints,
            cfg=cfg,mu=mu.values,
            delta=cfg.risk_aversion,prev=prev
        )
        if s is not None:
            sols.append(s)

    if not sols:
        raise RuntimeError("No feasible posterior EDaR solution.")

    best=min(sols,key=lambda x:x[2])
    w=best[0]
    w.index=R.columns
    return w,best


## 11. Volatility targeting

Uses a 126-day EWMA volatility estimate plus a fast 21-day trigger. If fast volatility exceeds twice long-run volatility, leverage is capped at 0.5x. Re-leveraging can be made slower in a production implementation.


In [ ]:
def ewma_vol(x,lam=0.94):
    x=np.asarray(x)
    v=np.var(x[:min(20,len(x))])
    for r in x:
        v=lam*v+(1-lam)*r*r
    return np.sqrt(v*252)

def target_leverage(x,cfg=CFG):
    long_vol=ewma_vol(x[-126:],cfg.ewma_lambda)
    fast_vol=np.std(x[-21:])*np.sqrt(252) if len(x)>=21 else long_vol
    k=np.clip(
        cfg.target_vol/max(long_vol,1e-8),
        cfg.min_leverage,
        cfg.max_leverage
    )
    if fast_vol>2*long_vol:
        k=min(k,0.50)
    return float(k),float(long_vol),float(fast_vol)


## 12. Execution model

Costs:
- linear turnover cost
- square-root ADV market impact
- daily short-borrow cost

This is intentionally a simple research model and should be calibrated against actual fills.


In [ ]:
def trading_cost(old_w,new_w,meta,cfg=CFG):
    trade=(new_w-old_w).abs()
    linear=trade.sum()*cfg.turnover_bps/1e4

    adv_dollars=meta.loc[trade.index,"ADV"].values
    trade_dollars=trade.values*1e9
    participation=trade_dollars/np.maximum(adv_dollars,1)
    impact=(cfg.impact_bps/1e4)*np.sum(
        trade.values*np.sqrt(np.maximum(participation,0))
    )

    borrow=(-new_w.clip(upper=0)).sum()*cfg.borrow_bps/1e4/252
    return float(linear+impact+borrow)


## 13. Walk-forward backtest

Each rebalance:
1. estimates beta and factor covariance from past data
2. solves beta/factor/sector-neutral min-EDaR
3. estimates the EDaR marginal gradient
4. forms the KKT-aware BL prior
5. generates independent views
6. solves posterior EDaR
7. applies volatility targeting
8. charges execution costs

No future data enter the rebalance decision.


In [ ]:
def walk_forward(R,benchmark,meta,cfg=CFG):
    dates=R.index
    prev=pd.Series(0.0,index=R.columns)
    current=prev.copy()
    lev=1.0
    last=-cfg.rebalance_days

    out=[]
    wh=[]
    levs=[]
    costs=[]

    B_constraints=exposure_matrix(meta).drop(columns=["Market"])

    for i in range(cfg.lookback,len(dates)-1):
        cost_today=0.0

        if i-last>=cfg.rebalance_days:
            train=R.iloc[i-cfg.lookback:i]

            beta=shrunk_betas(
                train,
                benchmark.iloc[i-cfg.lookback:i],
                cfg.beta_window
            )

            Sigma=factor_covariance(
                train,meta,cfg.cov_window
            )

            w0,_=min_edar(
                train,beta,B_constraints,cfg,prev
            )

            pi,_=kkt_prior(
                w0,train,beta,B_constraints,
                cfg.risk_aversion
            )

            P,q=independent_views(train,meta)

            mu,_,_=black_litterman(
                Sigma,pi,P,q,cfg.tau
            )

            w,_=posterior_edar(
                train,mu,beta,B_constraints,cfg,prev
            )

            shape=train@w
            lev,_,_=target_leverage(shape,cfg)

            cost_today=trading_cost(
                prev,w,meta,cfg
            )

            current=w
            prev=w.copy()
            last=i

        raw=float(R.iloc[i]@current)
        net=lev*raw-cost_today

        out.append(net)
        wh.append((lev*current).values)
        levs.append(lev)
        costs.append(cost_today)

    idx=dates[cfg.lookback:len(dates)-1]

    return {
        "returns":pd.Series(out,index=idx,name="Strategy"),
        "weights":pd.DataFrame(wh,index=idx,columns=R.columns),
        "leverage":pd.Series(levs,index=idx),
        "costs":pd.Series(costs,index=idx)
    }

# Full run can be computationally expensive for 500 names.
# bt = walk_forward(returns,benchmark,universe,CFG)


## 14. Performance statistics


In [ ]:
def max_dd(r):
    w=(1+r).cumprod()
    return float((w/w.cummax()-1).min())

def cvar(r,a=.05):
    q=np.quantile(r,a)
    return float(-r[r<=q].mean())

def stats(r):
    r=pd.Series(r).dropna()
    years=len(r)/252
    wealth=(1+r).cumprod()
    cagr=wealth.iloc[-1]**(1/years)-1
    vol=r.std()*np.sqrt(252)
    sharpe=r.mean()/r.std()*np.sqrt(252)
    ed,_=edar_value(r.values)
    return pd.Series({
        "CAGR":cagr,
        "Vol":vol,
        "Sharpe":sharpe,
        "MaxDD":max_dd(r),
        "EDaR":ed,
        "CVaR":cvar(r),
        "HitRate":(r>0).mean()
    })


## 15. Ablation study

Run identical walk-forward dates and costs for:
- minimum variance
- minimum EDaR
- EDaR + BL
- EDaR + BL + volatility targeting

The purpose is to test incremental contribution rather than to assume every layer helps.


In [ ]:
def ablation_table(results):
    return pd.DataFrame({
        name:stats(r)
        for name,r in results.items()
    })

# Example after running each architecture:
# results = {
#     "MinVar": r_minvar,
#     "MinEDaR": r_edar,
#     "EDaR+BL": r_bl,
#     "EDaR+BL+VolTarget": r_full,
# }
# ablation_table(results)


## 16. Stress tests


In [ ]:
def stress_pnl(w,meta,market=-0.08,tech=-0.10,energy=-0.10,rates=-0.04):
    shock=pd.Series(market,index=w.index)
    shock.loc[meta.index[meta.Sector=="Technology"]] += tech-market
    shock.loc[meta.index[meta.Sector=="Energy"]] += energy-market
    shock.loc[meta.index[meta.Sector=="Financials"]] += rates-market
    return float(w@shock)

def correlation_stress_vol(w,vol=.04,corr=.90):
    n=len(w)
    C=np.full((n,n),corr)
    np.fill_diagonal(C,1.0)
    S=(vol**2)*C
    return float(np.sqrt(w.values@S@w.values))

def stress_grid(w,meta):
    rows=[]
    scenarios=[
        ("Market crash",-0.08,-0.10,-0.10,-0.04),
        ("Technology crash",-0.03,-0.20,-0.02,-0.02),
        ("Energy shock",-0.03,-0.02,-0.20,-0.02),
        ("Rates shock",-0.03,-0.05,-0.02,-0.10)
    ]
    for name,m,t,e,r in scenarios:
        rows.append({
            "Scenario":name,
            "PnL":stress_pnl(w,meta,m,t,e,r)
        })
    return pd.DataFrame(rows).set_index("Scenario")


## 17. Monte Carlo block-bootstrap drawdown stress


In [ ]:
def monte_carlo_drawdowns(portfolio_returns,n_paths=2000,horizon=252,block=10,seed=123):
    rng=default_rng(seed)
    x=np.asarray(portfolio_returns)
    paths=[]
    for _ in range(n_paths):
        p=[]
        total=0
        while total<horizon:
            s=rng.integers(0,max(1,len(x)-block))
            c=x[s:s+block]
            p.extend(c)
            total+=len(c)
        paths.append(p[:horizon])

    mdd=[]
    for p in paths:
        w=np.cumprod(1+np.asarray(p))
        mdd.append(np.min(w/np.maximum.accumulate(w)-1))
    return np.asarray(mdd)


## 18. Parameter sensitivity

A robust strategy should not depend on one exact alpha, tau, gross cap or beta band.


In [ ]:
PARAM_GRID={
    "alpha":[0.025,0.05,0.10],
    "gross_limit":[1.5,2.0,2.5],
    "beta_band":[0.01,0.02,0.05],
    "tau":[0.025,0.05,0.10],
    "target_vol":[0.08,0.10,0.12]
}

def parameter_report():
    rows=[]
    for k,vals in PARAM_GRID.items():
        for v in vals:
            rows.append((k,v))
    return pd.DataFrame(rows,columns=["Parameter","Value"])

parameter_report()


## 19. Tail beta and exposure diagnostics


In [ ]:
def tail_beta(strategy,benchmark,threshold=-0.01):
    idx=benchmark<threshold
    x=benchmark[idx]
    y=strategy[idx]
    if len(x)<10:
        return np.nan
    return float(np.cov(y,x)[0,1]/np.var(x))

def exposure_diagnostics(weights):
    gross=weights.abs().sum(axis=1)
    largest=weights.abs().max(axis=1)
    effective=1/(weights.pow(2).sum(axis=1)+1e-12)
    return pd.DataFrame({
        "Gross":gross,
        "LargestPosition":largest,
        "EffectivePositions":effective
    })


## 20. Point-in-time live-data hook

For real research, supply:
- historical constituents including delisted names
- point-in-time sector classifications
- point-in-time market caps and ADV
- adjusted returns
- borrow availability and stock-loan fees
- point-in-time fundamentals

Do not substitute today's constituents into a historical backtest.


In [ ]:
def validate_data(R,meta):
    assert R.index.is_monotonic_increasing
    assert R.columns.is_unique
    assert R.columns.isin(meta.index).all()
    assert R.isna().mean().mean()<0.01
    assert (meta.loc[R.columns,"ADV"]>0).all()
    return True

validate_data(returns,universe)


## 21. Production research checklist

- [ ] Point-in-time universe and fundamentals
- [ ] Delisted securities
- [ ] Corporate-action-adjusted returns
- [ ] Historical ADV
- [ ] Borrow availability and borrow-rate history
- [ ] Factor covariance calibrated out of sample
- [ ] Purged/embargoed signal validation
- [ ] KKT-aware BL prior
- [ ] Beta/factor/sector neutrality bands
- [ ] Transaction-cost and market-impact calibration
- [ ] Correlation/beta/liquidity stress tests
- [ ] Parameter sensitivity
- [ ] Full ablation study
- [ ] Paper trading before deployment

**Solver:** CLARABEL is used for portability. For the full 500-name exponential-cone problem, MOSEK is the preferred production solver when available.
